# Thesis Visualizations

This notebook creates the figures used in the thesis. It uses the saved benchmark outputs and the implemented system architecture.



## Setup

The figures are built with Plotly. Static image export needs Kaleido.

The notebook uses the same blue and red colors as the evaluation notebook.


In [ ]:
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

OUT = Path('../docs/thesis_figures')
summary = pd.read_csv('../docs/evaluation_outputs/summary_metrics.csv')

COLORS = {
    'dark': '#0F172A',
    'navy': '#0B2545',
    'blue': '#1D4E89',
    'mid': '#4F86C6',
    'light': '#A9C6E8',
    'pale': '#EAF3FB',
    'line': '#CBD5E1',
    'grid': '#E2E8F0',
    'red': '#B23A48',
    'pale_red': '#FDE2E5',
    'gray': '#64748B',
    'white': '#FFFFFF'
}

def export(fig, name, width=1500, height=900):
    fig.write_image(str(OUT / name), width=width, height=height, scale=2)


## Formal Metric Definitions

Most benchmark metrics are rates. A rate is calculated as the number of passed applicable cases divided by the number of applicable cases. Failed cases stay in the denominator.

Aggregation Accuracy checks the primary aggregation step. It does not include conversational follow-up correction. Follow-up correction is measured separately by Follow-Up Consistency.

Groundedness is a benchmark check for evidence and domain consistency. It is useful, but it is not a proof that hallucinations are impossible.


## Benchmark Charts

The next cell rebuilds the benchmark charts from the audited CSV files.

The charts use only saved evaluation outputs. They do not recalculate or change the benchmark results.


In [ ]:
rate = summary[summary['metric'] != 'Average Latency'].copy()
rate['percent'] = rate['value'] * 100
rate['color'] = rate['percent'].apply(lambda v: COLORS['red'] if v < 80 else COLORS['blue'])
fig = make_subplots(rows=1, cols=2, column_widths=[0.78, 0.22], specs=[[{'type': 'xy'}, {'type': 'indicator'}]], subplot_titles=('Rate metrics', 'Latency'))
fig.add_trace(go.Bar(y=rate['metric'], x=rate['percent'], orientation='h', marker_color=rate['color'], text=rate['display_value'], textposition='outside', cliponaxis=False), row=1, col=1)
latency = summary[summary['metric'] == 'Average Latency'].iloc[0]
fig.add_trace(go.Indicator(mode='number', value=float(latency['value']), number={'suffix': 's', 'font': {'size': 38, 'color': COLORS['navy']}}, title={'text': 'Mean latency'}), row=1, col=2)
fig.update_layout(title={'text': 'Benchmark Metric Breakdown', 'x': 0.5, 'font': {'size': 28, 'color': COLORS['dark']}}, width=1500, height=850, paper_bgcolor=COLORS['white'], plot_bgcolor=COLORS['white'], margin={'l': 260, 'r': 80, 't': 110, 'b': 70}, font={'family': 'Arial, sans-serif', 'size': 15, 'color': COLORS['dark']}, showlegend=False)
fig.update_xaxes(range=[0, 108], title='Percent', gridcolor=COLORS['grid'], row=1, col=1)
fig.update_yaxes(autorange='reversed', row=1, col=1)
export(fig, 'metric_breakdown.png', 1500, 850)


## Architecture Diagram Helpers

The architecture figures use grouped subsystem containers. They show the implemented frontend, backend, planning, execution, validation, artifact, and persistence parts.



In [ ]:
def base_fig(width=1500, height=900, title=None):
    fig = go.Figure()
    fig.update_xaxes(range=[0, 100], visible=False)
    fig.update_yaxes(range=[0, 100], visible=False)
    fig.update_layout(width=width, height=height, paper_bgcolor=COLORS['white'], plot_bgcolor=COLORS['white'], margin={'l': 35, 'r': 35, 't': 80 if title else 35, 'b': 35}, font={'family': 'Arial, sans-serif', 'size': 15, 'color': COLORS['dark']})
    if title:
        fig.update_layout(title={'text': title, 'x': 0.5, 'font': {'size': 26, 'color': COLORS['dark']}})
    return fig

def container(fig, x0, y0, x1, y1, label, fill='#F8FBFF'):
    fig.add_shape(type='rect', x0=x0, y0=y0, x1=x1, y1=y1, line={'color': COLORS['line'], 'width': 2}, fillcolor=fill, layer='below', xref='x', yref='y')
    fig.add_annotation(x=x0+2, y=y1-3, text=f'<b>{label}</b>', showarrow=False, xanchor='left', yanchor='top', font={'size': 15, 'color': COLORS['navy']})

def box(fig, x, y, w, h, text, fill=None, color=None):
    fill = fill or COLORS['white']
    color = color or COLORS['blue']
    fig.add_shape(type='rect', x0=x, y0=y, x1=x+w, y1=y+h, line={'color': color, 'width': 2.4}, fillcolor=fill, xref='x', yref='y')
    fig.add_annotation(x=x+w/2, y=y+h/2, text=text, showarrow=False, align='center', font={'size': 14, 'color': COLORS['dark']})

def arrow(fig, x0, y0, x1, y1, color=None):
    fig.add_annotation(x=x1, y=y1, ax=x0, ay=y0, xref='x', yref='y', axref='x', ayref='y', showarrow=True, arrowhead=3, arrowsize=1.25, arrowwidth=3, arrowcolor=color or COLORS['blue'])


## Overall Architecture

This figure shows the main system layers in one view. It groups the frontend, backend API, planning layer, deterministic execution layer, validation layer, artifact and reporting layer, and persistence layer.


In [ ]:
fig = base_fig(1700, 980, 'Implemented System Architecture')
container(fig, 3, 60, 18, 85, 'Frontend Layer', COLORS['pale'])
container(fig, 23, 60, 38, 85, 'Backend Services', '#F7FAFF')
container(fig, 43, 60, 58, 85, 'Planning Layer', '#F7FAFF')
container(fig, 43, 28, 58, 52, 'Execution Layer', '#FBFDFF')
container(fig, 63, 60, 79, 85, 'Validation Layer', COLORS['pale'])
container(fig, 83, 60, 97, 85, 'Artifact and Reporting', COLORS['pale'])
container(fig, 23, 28, 38, 52, 'Persistence Layer', '#FBFDFF')
box(fig, 6, 73, 9, 5, 'Chat<br>UI')
box(fig, 6, 64, 9, 5, 'Upload<br>UI')
box(fig, 26, 73, 9, 5, 'FastAPI<br>Routes')
box(fig, 26, 64, 9, 5, 'Run<br>Service')
box(fig, 46, 73, 9, 5, 'Routing')
box(fig, 46, 64, 9, 5, 'Semantic<br>Planner')
box(fig, 46, 41, 9, 5, 'Plan<br>Executor')
box(fig, 46, 32, 9, 5, 'Fallback<br>Tools')
box(fig, 66, 73, 9, 5, 'Critic', color=COLORS['red'])
box(fig, 66, 64, 9, 5, 'Validation<br>Artifacts', color=COLORS['red'])
box(fig, 86, 73, 8, 5, 'Artifact<br>Adapter')
box(fig, 86, 64, 8, 5, 'Reports')
box(fig, 26, 41, 9, 5, 'SQLite<br>Store')
box(fig, 26, 32, 9, 5, 'Runtime<br>Tables')
arrow(fig, 15, 75.5, 26, 75.5)
arrow(fig, 35, 75.5, 46, 75.5)
arrow(fig, 55, 75.5, 66, 75.5)
arrow(fig, 75, 75.5, 86, 75.5)
arrow(fig, 94, 66.5, 94, 73.0)
arrow(fig, 50.5, 64, 50.5, 46)
arrow(fig, 55, 43.5, 66, 66.5, COLORS['red'])
arrow(fig, 15, 66.5, 26, 34.5)
arrow(fig, 35, 34.5, 46, 34.5)
arrow(fig, 35, 43.5, 46, 43.5)
export(fig, 'overall_system_architecture.png', 1700, 980)


## Additional Workflow Diagrams

This cell rebuilds the workflow diagrams used in the thesis. These diagrams describe the evaluated analytical pipeline, dataset upload flow, and evaluation process.


In [ ]:
diagram_specs = [
    ('analytical_execution_pipeline.png', 'Analytical Execution Pipeline', 1650, 800, [(3, 62, 20, 86, 'User Input', COLORS['pale']), (24, 55, 45, 88, 'Backend Orchestration', '#F7FAFF'), (49, 55, 67, 88, 'Planning Layer', '#F7FAFF'), (49, 18, 72, 47, 'Execution Layer', '#FBFDFF'), (76, 18, 96, 88, 'Validation and Reporting', COLORS['pale'])], [(6, 72, 11, 7, 'User<br>Question'), (27, 77, 13, 6, 'Run Creation'), (28, 66, 12, 6, 'Routing'), (29, 57, 13, 6, 'Dataset<br>Resolution'), (52, 76, 11, 6, 'Semantic<br>Planning'), (53, 64, 12, 6, 'Constraint<br>Checks'), (52, 35, 14, 6, 'Deterministic<br>Execution'), (55, 24, 14, 6, 'Computed<br>Evidence'), (80, 58, 12, 6, 'Artifact<br>Generation'), (81, 41, 11, 6, 'Report<br>Generation'), (80, 25, 12, 6, 'Saved<br>Response')], [(79, 75, 11, 6, 'Critic and<br>Validation')], [(17, 75, 27, 80), (40, 80, 52, 79), (40, 69, 53, 67), (58, 64, 59, 41), (90, 75, 86, 64), (86, 58, 86, 47), (86, 41, 86, 31), (80, 28, 16, 72)], [(66, 38, 79, 78)]),
    ('multi_dataset_branching.png', 'Multi-Dataset Branching', 1650, 850, [(4, 56, 23, 87, 'Linked Datasets', COLORS['pale']), (28, 56, 47, 88, 'Dataset Registry', '#F7FAFF'), (52, 10, 76, 88, 'Isolated Branch Scopes', '#FBFDFF'), (81, 31, 97, 82, 'Synthesis', COLORS['pale'])], [(7, 76, 11, 6, 'Dataset A'), (9, 66, 11, 6, 'Dataset B'), (7, 58, 13, 6, 'Dataset C'), (31, 78, 13, 6, 'Profiles'), (32, 66, 13, 6, 'Scoring'), (31, 58, 14, 6, 'Selected<br>Dataset IDs'), (55, 75, 14, 6, 'Branch A<br>Execution'), (58, 55, 14, 6, 'Branch B<br>Execution'), (55, 35, 14, 6, 'Branch C<br>Execution'), (61, 18, 12, 6, 'Evidence<br>Packages'), (84, 68, 10, 6, 'Comparative<br>Synthesis'), (84, 50, 10, 6, 'Validation'), (84, 36, 11, 6, 'Final<br>Answer')], [], [(19, 76, 31, 81), (20, 66, 32, 69), (20, 59, 31, 61), (44, 61, 55, 78), (44, 61, 58, 58), (44, 61, 55, 38), (66, 75, 67, 24), (69, 55, 67, 24), (66, 35, 67, 24), (73, 21, 84, 71), (89, 50, 89, 42)], [(89, 68, 89, 56)]),
    ('dataset_upload_lifecycle.png', 'Dataset Upload Lifecycle', 1500, 820, [(4, 54, 24, 84, 'Frontend', COLORS['pale']), (29, 54, 48, 84, 'FastAPI Upload', '#F7FAFF'), (53, 28, 73, 84, 'Preparation', '#FBFDFF'), (78, 28, 96, 84, 'Stored Data Source', COLORS['pale'])], [(7, 72, 13, 6, 'CSV Upload<br>Form'), (9, 60, 12, 6, 'Upload<br>Proxy'), (32, 70, 12, 6, 'Multipart<br>Request'), (32, 60, 13, 6, 'File and<br>Header Checks'), (56, 72, 12, 6, 'File<br>Storage'), (58, 58, 12, 6, 'Profiling'), (56, 43, 14, 6, 'Runtime<br>SQLite Table'), (81, 70, 11, 6, 'Data Source<br>Record'), (82, 55, 11, 6, 'Profile<br>Metadata'), (81, 40, 12, 6, 'Investigation<br>Link')], [], [(19, 75, 32, 73), (20, 63, 32, 63), (44, 70, 56, 75), (44, 60, 58, 61), (64, 58, 63, 49), (70, 46, 81, 43), (68, 75, 81, 73), (68, 61, 82, 58)], []),
    ('evaluation_pipeline.png', 'Deterministic Evaluation Pipeline', 1500, 770, [(4, 58, 23, 84, 'Benchmark Input', COLORS['pale']), (29, 50, 51, 86, 'System Run', '#F7FAFF'), (57, 35, 77, 86, 'Validation', '#FBFDFF'), (82, 44, 96, 82, 'Outputs', COLORS['pale'])], [(7, 72, 12, 6, 'Benchmark<br>Case'), (8, 62, 13, 6, 'Fixed<br>Dataframe'), (32, 76, 12, 6, 'Dataset<br>Scope'), (33, 64, 12, 6, 'Planner'), (33, 53, 13, 6, 'Deterministic<br>Execution'), (60, 75, 12, 6, 'Pandas<br>Reference'), (61, 61, 12, 6, 'Metric<br>Checks'), (60, 48, 13, 6, 'Failure<br>Recording'), (84, 69, 10, 6, 'CSV<br>Outputs'), (84, 56, 10, 6, 'Figures'), (84, 45, 10, 6, 'Report<br>Tables')], [], [(19, 75, 32, 79), (20, 64, 33, 56), (39, 76, 39, 70), (39, 64, 39, 59), (46, 56, 61, 64), (66, 75, 66, 67), (73, 64, 84, 72), (73, 51, 84, 48)], [(66, 61, 66, 54)])
]

for name, title, width, height, containers, boxes, red_boxes, arrows, red_arrows in diagram_specs:
    fig = base_fig(width, height, title)
    for item in containers:
        container(fig, *item)
    for item in boxes:
        box(fig, *item)
    for item in red_boxes:
        box(fig, *item, color=COLORS['red'])
    for item in arrows:
        arrow(fig, *item)
    for item in red_arrows:
        arrow(fig, *item, COLORS['red'])
    export(fig, name, width, height)


## KPI and Multi-Dataset Support Diagrams

This cell rebuilds diagrams for KPI reasoning, follow-up correction, dataset scoring, and branch-scoped execution.


In [ ]:
additional_specs = [
    ('kpi_reasoning_flow.png', 'Business KPI Reasoning Flow', [(4, 58, 24, 84, 'Question', COLORS['pale']), (29, 48, 49, 86, 'Business Semantics', '#F7FAFF'), (54, 26, 73, 86, 'Deterministic KPI Logic', '#FBFDFF'), (79, 37, 96, 82, 'Evidence Output', COLORS['pale'])], [(8, 70, 12, 6, 'Business<br>Question'), (32, 75, 12, 6, 'Intent<br>Detection'), (33, 62, 13, 6, 'Metric<br>Mapping'), (32, 51, 14, 6, 'Constraint<br>Locking'), (57, 74, 12, 6, 'KPI<br>Registry'), (58, 59, 12, 6, 'Formula<br>Execution'), (57, 42, 14, 6, 'Grouped KPI<br>Rows'), (82, 70, 10, 6, 'Validation'), (82, 55, 11, 6, 'Artifacts'), (82, 42, 11, 6, 'Grounded<br>Synthesis')], [], [(20, 73, 32, 78), (44, 75, 57, 77), (45, 64, 58, 62), (45, 53, 58, 46), (64, 74, 64, 65), (64, 59, 64, 48), (88, 70, 88, 61), (88, 55, 88, 48)], [(71, 45, 82, 73)]),
    ('followup_correction_flow.png', 'Follow-Up Correction Flow', [(4, 56, 24, 84, 'Conversation', COLORS['pale']), (29, 48, 52, 86, 'Context Recovery', '#F7FAFF'), (57, 43, 75, 86, 'Patch Attempt', '#FBFDFF'), (80, 43, 96, 82, 'Known Limit', COLORS['pale_red'])], [(7, 72, 13, 6, 'Follow-up<br>Question'), (32, 76, 13, 6, 'Recent Plan<br>Metadata'), (33, 62, 12, 6, 'Artifact<br>Context'), (33, 51, 13, 6, 'Explicit<br>Override'), (60, 74, 11, 6, 'Plan<br>Patching'), (61, 60, 12, 6, 'Re-execution'), (60, 47, 13, 6, 'New<br>Artifacts'), (83, 68, 10, 6, 'Validation'), (83, 53, 11, 7, 'One audited<br>failure')], [], [(20, 75, 32, 79), (44, 76, 60, 77), (45, 64, 60, 63), (45, 54, 60, 50), (66, 74, 67, 66), (67, 60, 67, 53)], [(73, 50, 83, 71), (88, 68, 88, 60)]),
    ('dataset_resolution_scoring.png', 'Dataset Resolution and Scoring', [(4, 56, 23, 84, 'Linked Datasets', COLORS['pale']), (29, 39, 51, 86, 'Registry Signals', '#F7FAFF'), (57, 45, 75, 84, 'Scoring', '#FBFDFF'), (80, 45, 96, 82, 'Decision', COLORS['pale'])], [(7, 73, 12, 6, 'Columns'), (8, 63, 12, 6, 'Samples'), (32, 76, 14, 6, 'Semantic<br>Roles'), (33, 64, 13, 6, 'Runtime<br>Availability'), (32, 52, 14, 6, 'Lineage<br>Metadata'), (60, 72, 11, 6, 'Mention<br>Match'), (61, 59, 12, 6, 'Metric and<br>Role Match'), (60, 48, 13, 6, 'Sample<br>Value Match'), (83, 70, 10, 6, 'Single<br>Dataset'), (83, 58, 10, 6, 'Several<br>Datasets'), (83, 47, 10, 6, 'Clarify')], [], [(19, 74, 32, 79), (20, 65, 33, 67), (46, 77, 60, 75), (46, 64, 61, 62), (46, 54, 60, 51), (71, 75, 83, 73), (72, 62, 83, 61), (72, 51, 83, 50)], []),
    ('branch_execution_scopes.png', 'Dataset-Scoped Execution Branches', [(4, 55, 22, 82, 'Selected Datasets', COLORS['pale']), (28, 17, 51, 88, 'Branch A Scope', '#F7FAFF'), (54, 17, 77, 80, 'Branch B Scope', '#FBFDFF'), (82, 32, 96, 82, 'Comparative Output', COLORS['pale'])], [(7, 70, 11, 6, 'Dataset A'), (7, 59, 11, 6, 'Dataset B'), (31, 76, 12, 6, 'Local<br>Context'), (32, 61, 13, 6, 'Execution'), (32, 46, 13, 6, 'Findings'), (31, 31, 14, 6, 'Artifacts'), (57, 68, 12, 6, 'Local<br>Context'), (58, 53, 13, 6, 'Execution'), (58, 39, 13, 6, 'Findings'), (57, 25, 14, 6, 'Artifacts'), (84, 68, 10, 6, 'Evidence<br>Packages'), (84, 51, 10, 6, 'Synthesis'), (84, 38, 10, 6, 'Response')], [], [(18, 73, 31, 79), (18, 62, 57, 71), (38, 76, 38, 67), (38, 61, 38, 52), (38, 46, 38, 37), (64, 68, 64, 59), (64, 53, 64, 45), (64, 39, 64, 31), (45, 34, 84, 71), (70, 28, 84, 71), (89, 68, 89, 57), (89, 51, 89, 44)], [])
]

for name, title, containers, boxes, red_boxes, arrows, red_arrows in additional_specs:
    fig = base_fig(1500, 820, title)
    for item in containers:
        container(fig, *item)
    for item in boxes:
        box(fig, *item)
    for item in red_boxes:
        box(fig, *item, color=COLORS['red'])
    for item in arrows:
        arrow(fig, *item)
    for item in red_arrows:
        arrow(fig, *item, COLORS['red'])
    export(fig, name, 1500, 820)
